# 05b - Test Group Integration for Trigger Detection

This notebook tests the ingredient grouping system and group-aware trigger detection.

**Scientific Rationale:**
If "cheddar cheese" is a trigger, other cheese group members (mozzarella, parmesan) 
are likely triggers too due to shared chemical properties (lactose, casein).

**Date:** February 2026  
**Author:** SafePlate Team

## Setup

In [3]:
import sys
from pathlib import Path

# Add backend to path
project_root = Path.cwd().parent
backend_path = project_root / "backend"
sys.path.insert(0, str(backend_path))

print(f"Project root: {project_root}")

Project root: /Users/adarpalaoglu/Documents/cs310-safeplate


In [4]:
# Import modules
from safeplate.ingredient_groups import IngredientGroupManager, get_group_manager
from safeplate.embedding_loader import EmbeddingLoader
from safeplate.recommendation_engine import RecommendationEngine
from safeplate.group_aware_trigger_detector import (
    GroupAwareTriggerDetector,
    TriggerDetectionResult
)

import pandas as pd
import numpy as np

ImportError: cannot import name 'TriggerDetectionResult' from 'safeplate.group_aware_trigger_detector' (/Users/adarpalaoglu/Documents/cs310-safeplate/backend/safeplate/group_aware_trigger_detector.py)

## Test 1: Load Ingredient Groups and Show Statistics

In [ ]:
# Initialize group manager
groups_path = project_root / "data" / "processed" / "ingredient_groups.csv"
gm = IngredientGroupManager(str(groups_path))
gm.load()

# Show statistics
stats = gm.get_group_statistics()

print("=" * 60)
print("INGREDIENT GROUP STATISTICS")
print("=" * 60)
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Show groups with multiple members (useful for propagation)
multi_member = gm.get_groups_with_multiple_members()

print(f"\nGroups with multiple members: {len(multi_member)}")
print("\nTop 10 largest groups:")

sorted_groups = sorted(multi_member.items(), key=lambda x: len(x[1]), reverse=True)
for group, members in sorted_groups[:10]:
    print(f"  {group}: {len(members)} members")

## Test 2: Group Lookup Functions

In [ ]:
# Test get_group()
test_ingredients = [
    "cheddar cheese",
    "mozzarella",
    "parmesan",
    "tomato",
    "cherry tomato",
    "milk",
    "wheat flour",
    "unicorn_meat"  # Should return None
]

print("=" * 60)
print("TEST: get_group()")
print("=" * 60)

for ing in test_ingredients:
    group = gm.get_group(ing)
    print(f"  {ing:<20} -> {group}")

In [ ]:
# Test get_group_members()
print("\n" + "=" * 60)
print("TEST: get_group_members()")
print("=" * 60)

for group in ["cheese", "tomato", "milk", "wheat", "peanut"]:
    members = gm.get_group_members(group)
    print(f"\n{group.upper()} ({len(members)} members):")
    for m in members[:8]:
        print(f"    - {m}")
    if len(members) > 8:
        print(f"    ... and {len(members) - 8} more")

In [ ]:
# Test are_in_same_group()
print("\n" + "=" * 60)
print("TEST: are_in_same_group()")
print("=" * 60)

pairs = [
    ("cheddar cheese", "mozzarella"),
    ("tomato", "cherry tomato"),
    ("tomato", "cheese"),
    ("wheat flour", "bread flour"),
    ("milk", "cream"),
]

for ing1, ing2 in pairs:
    same = gm.are_in_same_group(ing1, ing2)
    print(f"  {ing1:<20} & {ing2:<20}: {same}")

In [ ]:
# Test expand_to_group_members()
print("\n" + "=" * 60)
print("TEST: expand_to_group_members()")
print("=" * 60)

triggers = ["cheddar cheese", "tomato"]
expanded = gm.expand_to_group_members(triggers)

print(f"\nOriginal triggers: {triggers}")
print(f"Expanded count: {len(expanded)}")
print("\nExpanded list (sample):")
for ing in sorted(expanded)[:15]:
    print(f"  - {ing}")
if len(expanded) > 15:
    print(f"  ... and {len(expanded) - 15} more")

## Test 3: Group-Aware Recommendation Filtering

In [ ]:
# Initialize recommendation engine
embedding_path = project_root / "data" / "processed" / "embeddings" / "ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w10_wl80_r80_e10.txt"
loader = EmbeddingLoader(str(embedding_path))
engine = RecommendationEngine(loader)

In [ ]:
# Compare: Regular filtering vs Group-aware filtering
print("=" * 70)
print("COMPARISON: Regular vs Group-Aware Filtering")
print("=" * 70)

query = "milk"
individual_trigger = ["cheddar cheese"]

# Regular filtering - only excludes cheddar cheese
print(f"\nQuery: '{query}'")
print(f"Trigger: {individual_trigger}")

print("\n--- REGULAR FILTERING (exclude only 'cheddar cheese') ---")
regular_results = engine.find_substitutes(query, triggers=individual_trigger, top_n=10)
for i, (ing, score) in enumerate(regular_results, 1):
    marker = "[CHEESE]" if gm.get_group(ing) == "cheese" else ""
    print(f"  {i:2}. {ing:<30} ({score:.4f}) {marker}")

In [ ]:
# Group-aware filtering - excludes all cheese group members
print("\n--- GROUP-AWARE FILTERING (exclude entire 'cheese' group) ---")
group_results = engine.find_substitutes_group_aware(
    query_ingredient=query,
    trigger_ingredients=individual_trigger,
    trigger_groups=["cheese"],
    top_n=10,
    group_manager=gm
)

for i, (ing, score) in enumerate(group_results, 1):
    group = gm.get_group(ing)
    print(f"  {i:2}. {ing:<30} ({score:.4f}) [group: {group}]")

In [ ]:
# Test expand_triggers option
print("\n" + "=" * 70)
print("TEST: Auto-expand triggers to group members")
print("=" * 70)

results, excluded = engine.find_substitutes_expand_triggers(
    query_ingredient="milk",
    trigger_ingredients=["cheddar cheese", "tomato"],
    expand_to_groups=True,
    top_n=10,
    group_manager=gm
)

print(f"\nOriginal triggers: ['cheddar cheese', 'tomato']")
print(f"Total excluded (after expansion): {len(excluded)}")
print("\nTop 10 substitutes:")
for i, (ing, score) in enumerate(results, 1):
    print(f"  {i:2}. {ing:<30} ({score:.4f})")

In [ ]:
# Show what gets filtered
print("\n--- WHAT GETS FILTERED ---")
cheese_excluded = [e for e in excluded if gm.get_group(e) == "cheese"]
tomato_excluded = [e for e in excluded if gm.get_group(e) == "tomato"]

print(f"\nCheese group members excluded ({len(cheese_excluded)}):")
for ing in sorted(cheese_excluded)[:10]:
    print(f"  - {ing}")
if len(cheese_excluded) > 10:
    print(f"  ... and {len(cheese_excluded) - 10} more")

print(f"\nTomato group members excluded ({len(tomato_excluded)}):")
for ing in sorted(tomato_excluded)[:10]:
    print(f"  - {ing}")
if len(tomato_excluded) > 10:
    print(f"  ... and {len(tomato_excluded) - 10} more")

## Test 4: Group Evidence Propagation

In [ ]:
# Initialize group-aware trigger detector
detector = GroupAwareTriggerDetector(group_manager=gm)

In [ ]:
# Simulate trigger detection results from case-crossover analysis
# Format: (ingredient, odds_ratio, p_value)
simulated_triggers = [
    # Strong cheese trigger
    ("cheddar cheese", 3.2, 0.001),
    # Weak cheese signal (not significant alone)
    ("mozzarella", 1.8, 0.08),
    # Medium tomato trigger
    ("tomato", 2.1, 0.045),
    # Strong individual trigger (no group)
    ("shrimp", 4.5, 0.0005),
    # Weak signal
    ("garlic", 1.3, 0.15),
]

print("=" * 70)
print("SIMULATED CASE-CROSSOVER RESULTS (Input)")
print("=" * 70)
print(f"{'Ingredient':<20} {'Odds Ratio':>12} {'P-value':>12} {'Group':>15}")
print("-" * 60)
for ing, or_, p in simulated_triggers:
    group = gm.get_group(ing) or "(none)"
    print(f"{ing:<20} {or_:>12.2f} {p:>12.4f} {group:>15}")

In [ ]:
# Run group-aware trigger detection
result = detector.propagate_trigger_evidence(
    ingredient_triggers=simulated_triggers,
    propagation_discount=0.7,
    min_group_evidence=0.5
)

print("\n" + "=" * 70)
print("GROUP-AWARE TRIGGER DETECTION RESULTS")
print("=" * 70)

In [ ]:
# Show individual triggers
print("\n--- INDIVIDUAL TRIGGERS (Direct Evidence) ---")
print(f"{'Ingredient':<20} {'OR':>8} {'P-value':>10} {'Group':>12} {'Confidence':>12}")
print("-" * 65)

for t in result.individual_triggers:
    group = t.group or "(none)"
    print(f"{t.ingredient:<20} {t.odds_ratio:>8.2f} {t.p_value:>10.4f} {group:>12} {t.confidence:>12}")

In [ ]:
# Show group-level aggregation
print("\n--- GROUP-LEVEL AGGREGATION ---")
print(f"{'Group':<15} {'Members':>8} {'Sig.':>6} {'Avg OR':>10} {'Min P':>10} {'Strength':>10} {'Conf.':>10}")
print("-" * 75)

for g in result.group_triggers:
    print(f"{g.group:<15} {g.member_count:>8} {g.significant_members:>6} {g.avg_odds_ratio:>10.2f} {g.min_p_value:>10.4f} {g.evidence_strength:>10.2f} {g.confidence:>10}")

In [ ]:
# Show propagated triggers (the key insight!)
print("\n--- PROPAGATED TRIGGERS (Inferred from Group Evidence) ---")
print(f"{'Ingredient':<25} {'OR':>8} {'P-value':>10} {'Group':>12} {'Source':>20}")
print("-" * 80)

for t in result.propagated_triggers[:15]:  # Show first 15
    source = t.propagated_from or "N/A"
    print(f"{t.ingredient:<25} {t.odds_ratio:>8.2f} {t.p_value:>10.4f} {t.group:>12} {source:>20}")

if len(result.propagated_triggers) > 15:
    print(f"\n... and {len(result.propagated_triggers) - 15} more propagated triggers")

In [ ]:
# Show the complete picture for cheese group
print("\n" + "=" * 70)
print("CHEESE GROUP ANALYSIS")
print("=" * 70)

cheese_individual = [t for t in result.individual_triggers if t.group == "cheese"]
cheese_propagated = [t for t in result.propagated_triggers if t.group == "cheese"]

print(f"\nDirect evidence: {len(cheese_individual)} ingredients")
for t in cheese_individual:
    print(f"  - {t.ingredient}: OR={t.odds_ratio:.2f}, p={t.p_value:.4f} ({t.confidence})")

print(f"\nPropagated evidence: {len(cheese_propagated)} ingredients")
print("(These are inferred to be potential triggers based on group membership)")
for t in cheese_propagated[:10]:
    print(f"  - {t.ingredient}: OR={t.odds_ratio:.2f} (propagated from {t.propagated_from})")
if len(cheese_propagated) > 10:
    print(f"  ... and {len(cheese_propagated) - 10} more")

In [ ]:
# Get all trigger ingredients
all_triggers = result.get_all_trigger_ingredients()
high_conf = result.get_high_confidence_triggers()

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total trigger ingredients identified: {len(all_triggers)}")
print(f"  - Direct evidence: {len(result.individual_triggers)}")
print(f"  - Propagated from groups: {len(result.propagated_triggers)}")
print(f"  - High confidence: {len(high_conf)}")

## Test 5: Verify Existing Tests Still Work

In [ ]:
# Verify that original recommendation engine methods still work
print("=" * 70)
print("REGRESSION TEST: Original Methods")
print("=" * 70)

# Test get_similar_ingredients (should still work)
similar = engine.get_similar_ingredients("tomato", top_n=5)
print("\nget_similar_ingredients('tomato', top_n=5):")
for ing, score in similar:
    print(f"  - {ing}: {score:.4f}")

# Test find_substitutes (should still work)
subs = engine.find_substitutes("milk", triggers=["cream"], top_n=5)
print("\nfind_substitutes('milk', triggers=['cream'], top_n=5):")
for ing, score in subs:
    print(f"  - {ing}: {score:.4f}")

print("\n[OK] Original methods working correctly!")

## Summary

### What We Built

1. **IngredientGroupManager** - O(1) lookups for ingredient groups
   - `get_group()` - Find group for an ingredient
   - `get_group_members()` - Get all members of a group
   - `expand_to_group_members()` - Expand triggers to include related ingredients

2. **Group-Aware Recommendation Engine**
   - `find_substitutes_group_aware()` - Exclude entire groups, not just individuals
   - `find_substitutes_expand_triggers()` - Auto-expand triggers to group members

3. **GroupAwareTriggerDetector**
   - `propagate_trigger_evidence()` - Infer triggers from group membership
   - Combines direct evidence with group-level aggregation

### Key Insight

If "cheddar cheese" (OR=3.2, p=0.001) is detected as a trigger:
- Traditional approach: Only avoid cheddar cheese
- Group-aware approach: Avoid ALL cheese (105 ingredients)

This is critical for dietary conditions where the underlying cause
(lactose, casein, etc.) is shared across the entire food group.

In [ ]:
# Final output
print("\n" + "=" * 70)
print("GROUP INTEGRATION TEST COMPLETE")
print("=" * 70)
print(f"Ingredient groups loaded: {gm.num_groups}")
print(f"Multi-member groups: {len(gm.get_groups_with_multiple_members())}")
print(f"Recommendation engine: {engine.num_ingredients} ingredients")
print(f"Group-aware detection: Ready for Step 2 integration")